# Train statistical models


In [ ]:
from pathlib import Path
from time import perf_counter

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import your
from scipy.special import expit
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score, average_precision_score, f1_score, log_loss,
    precision_recall_fscore_support, roc_auc_score
)
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.neural_network import MLPClassifier
from sklearn.kernel_approximation import RBFSampler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from tqdm.auto import tqdm


## Set input and output paths


In [ ]:
filterbank_file_path = Path('/hercules/results/akazantsev/filterbank_files/B0531+21/B0531+21_59000_48386.fil')
run_dir = Path('/u/akazantsev/ml_dl_for_rfi_mitigation/runs/punch_rfi_reproduction')
base_name = 'B0531+21_59000_48386'

metadata_path = run_dir / f'{base_name}_training_metadata.csv'
split_path = run_dir / f'{base_name}_training_split_indices.npz'
comparison_path = run_dir / 'statistical_model_comparison.csv'
importance_path = run_dir / 'statistical_feature_importance.csv'
bundle_path = run_dir / 'best_statistical_model.joblib'

if not filterbank_file_path.is_file():
    raise FileNotFoundError(f'Filterbank file not found: {filterbank_file_path}')
if not metadata_path.is_file():
    raise FileNotFoundError(f'Training metadata not found: {metadata_path}')
if not split_path.is_file():
    raise FileNotFoundError(f'Training split not found: {split_path}')


## Load and check training data


In [ ]:
feature_columns = [
    'mean_o', 'std_o', 'skew_o', 'kurt_o',
    'mean_n', 'std_n', 'skew_n', 'kurt_n',
    'mean_o_ratio', 'std_o_ratio', 'skew_o_ratio', 'kurt_o_ratio',
    'mean_n_ratio', 'std_n_ratio', 'skew_n_ratio', 'kurt_n_ratio'
]

metadata = pd.read_csv(metadata_path)
required_columns = {'sample_index', 'label', *feature_columns}
if not required_columns.issubset(metadata.columns):
    raise ValueError('Training metadata has an unexpected schema')
if metadata['sample_index'].to_numpy().tolist() != list(range(len(metadata))):
    raise ValueError('Training metadata rows do not match sample indices')
if metadata['label'].value_counts().to_dict() != {'BGN': 10_000, 'NBRFI': 10_000}:
    raise ValueError('Training metadata has an unexpected class balance')

features = metadata.loc[:, feature_columns].apply(pd.to_numeric, errors='raise')
if np.isinf(features.to_numpy()).any():
    raise ValueError('Training features contain infinite values')
targets = (metadata['label'].to_numpy() == 'NBRFI').astype(np.int64)

with np.load(split_path) as split_file:
    train_indices = split_file['train_indices']
    validation_indices = split_file['validation_indices']
    test_indices = split_file['test_indices']

if (len(train_indices), len(validation_indices), len(test_indices)) != (16_000, 2_000, 2_000):
    raise ValueError('Training split has unexpected sizes')
all_indices = np.concatenate([train_indices, validation_indices, test_indices])
if len(np.unique(all_indices)) != len(metadata) or set(all_indices) != set(range(len(metadata))):
    raise ValueError('Training split does not cover each row once')

X_train = features.iloc[train_indices]
X_validation = features.iloc[validation_indices]
X_test = features.iloc[test_indices]
y_train = targets[train_indices]
y_validation = targets[validation_indices]
y_test = targets[test_indices]

if not all(np.unique(values).tolist() == [0, 1] for values in (y_train, y_validation, y_test)):
    raise ValueError('A split does not contain both classes')

print('Training rows:', len(X_train))
print('Validation rows:', len(X_validation))
print('Test rows:', len(X_test))
print('Feature count:', len(feature_columns))


## Plot feature correlation


In [ ]:
correlation = X_train.corr()
fig, axis = plt.subplots(figsize=(10, 8))
image = axis.imshow(correlation, vmin=-1, vmax=1, cmap='coolwarm')
axis.set_xticks(range(len(feature_columns)), feature_columns, rotation=90)
axis.set_yticks(range(len(feature_columns)), feature_columns)
axis.set_title('Training feature correlation')
fig.colorbar(image, ax=axis, label='Correlation')
fig.tight_layout()
plt.show()


## Define model family


In [ ]:
models = {
    'SGD_LogReg': Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value=0.0)),
        ('clf', SGDClassifier(loss='log_loss', alpha=1e-4, max_iter=2_000, tol=1e-3, random_state=42))
    ]),
    'SGD_LinearSVM': Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value=0.0)),
        ('clf', SGDClassifier(loss='hinge', alpha=1e-4, max_iter=2_000, tol=1e-3, random_state=42))
    ]),
    'Poly2_LogReg': Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value=0.0)),
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),
        ('clf', SGDClassifier(loss='log_loss', alpha=1e-4, max_iter=2_000, tol=1e-3, random_state=42))
    ]),
    'RBFapprox_LogReg': Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value=0.0)),
        ('rbf', RBFSampler(gamma=1.0, n_components=800, random_state=42)),
        ('clf', SGDClassifier(loss='log_loss', alpha=1e-4, max_iter=2_000, tol=1e-3, random_state=42))
    ]),
    'HistGB': Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value=0.0)),
        ('clf', HistGradientBoostingClassifier(learning_rate=0.05, max_depth=6, max_iter=300, random_state=42))
    ]),
    'MLP': Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value=0.0)),
        ('clf', MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', alpha=1e-4, learning_rate_init=1e-3, max_iter=200, random_state=42))
    ])
}


## Define evaluation functions


In [ ]:
def probability_scores(model, values):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(values)[:, 1]
    return expit(model.decision_function(values))


def threshold_by_f1(target, scores):
    thresholds = np.linspace(0.01, 0.99, 99)
    f1_values = [f1_score(target, scores >= threshold, zero_division=0) for threshold in thresholds]
    return float(thresholds[int(np.argmax(f1_values))])


def evaluate_binary(target, scores, threshold):
    prediction = (scores >= threshold).astype(np.int64)
    precision, recall, f1, _ = precision_recall_fscore_support(
        target, prediction, average='binary', zero_division=0
    )
    return {
        'accuracy': float(accuracy_score(target, prediction)),
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1),
        'pr_auc': float(average_precision_score(target, scores)),
        'roc_auc': float(roc_auc_score(target, scores)),
        'log_loss': float(log_loss(target, scores, labels=[0, 1]))
    }


## Select three features


In [ ]:
# Use a separate training partition for feature selection.
selector = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
selection_fit_positions, selection_eval_positions = next(selector.split(X_train, y_train))
X_selection_fit = X_train.iloc[selection_fit_positions]
X_selection_eval = X_train.iloc[selection_eval_positions]
y_selection_fit = y_train[selection_fit_positions]
y_selection_eval = y_train[selection_eval_positions]

importance_rows = []
top_features_by_model = {}

for model_name, model in tqdm(models.items(), desc='Selecting features'):
    selection_model = clone(model)
    selection_model.fit(X_selection_fit, y_selection_fit)
    importance = permutation_importance(
        selection_model,
        X_selection_eval,
        y_selection_eval,
        scoring='average_precision',
        n_repeats=10,
        random_state=42,
        n_jobs=1
    )
    importance_table = pd.DataFrame({
        'model': model_name,
        'feature': feature_columns,
        'importance_mean': importance.importances_mean,
        'importance_std': importance.importances_std
    }).sort_values('importance_mean', ascending=False, kind='stable')
    importance_table['rank'] = range(1, len(importance_table) + 1)
    importance_rows.append(importance_table)
    top_features_by_model[model_name] = importance_table.head(3)['feature'].tolist()

importance_results = pd.concat(importance_rows, ignore_index=True)
importance_results.to_csv(importance_path, index=False)
display(importance_results[importance_results['rank'] <= 3])


## Train and compare models


In [ ]:
comparison_rows = []
fitted_top3_models = {}

for model_name, model in tqdm(models.items(), desc='Training models'):
    for variant, columns in (('full', feature_columns), ('top3', top_features_by_model[model_name])):
        fitted_model = clone(model)
        start_time = perf_counter()
        fitted_model.fit(X_train.loc[:, columns], y_train)
        fit_seconds = perf_counter() - start_time

        validation_scores = probability_scores(fitted_model, X_validation.loc[:, columns])
        threshold = threshold_by_f1(y_validation, validation_scores)
        validation_metrics = evaluate_binary(y_validation, validation_scores, threshold)

        test_scores = probability_scores(fitted_model, X_test.loc[:, columns])
        test_metrics = evaluate_binary(y_test, test_scores, threshold)

        comparison_rows.append({
            'model': model_name,
            'variant': variant,
            'feature_count': len(columns),
            'selected_features': ', '.join(columns),
            'fit_seconds': fit_seconds,
            'threshold': threshold,
            **{f'validation_{key}': value for key, value in validation_metrics.items()},
            **{f'test_{key}': value for key, value in test_metrics.items()}
        })

        if variant == 'top3':
            fitted_top3_models[model_name] = {
                'pipeline': fitted_model,
                'feature_columns': columns,
                'threshold': threshold,
                'validation_metrics': validation_metrics,
                'test_metrics': test_metrics
            }

comparison_results = pd.DataFrame(comparison_rows)
comparison_results.to_csv(comparison_path, index=False)
display(comparison_results.sort_values(['variant', 'test_f1'], ascending=[True, False]))
print('Comparison:', comparison_path)
print('Importance:', importance_path)


## Select inspection model


In [ ]:
inspection_model_name = None
inspection_segment_index = None

if inspection_model_name not in fitted_top3_models:
    raise ValueError('Set inspection_model_name to one model in the comparison table')
if not isinstance(inspection_segment_index, int) or inspection_segment_index < 0:
    raise ValueError('Set inspection_segment_index to a non-negative integer')


## Inspect a filterbank segment


In [ ]:
def compute_stats(values, axis=None):
    values = np.asarray(values)
    mean = values.mean(axis=axis, keepdims=True)
    std = values.std(axis=axis, keepdims=True)
    centered = values - mean
    std_safe = np.where(std == 0, 1.0, std)
    skew = (centered ** 3).mean(axis=axis, keepdims=True) / (std_safe ** 3)
    kurt = (centered ** 4).mean(axis=axis, keepdims=True) / (std_safe ** 4) - 3.0
    return mean.squeeze(), std.squeeze(), skew.squeeze(), kurt.squeeze()


def normalize_per_channel(segment):
    channel_min = segment.min(axis=1, keepdims=True)
    channel_max = segment.max(axis=1, keepdims=True)
    channel_range = channel_max - channel_min
    channel_range[channel_range == 0] = 1.0
    return 256.0 * (segment - channel_min) / channel_range


def compute_ratio(channel_values, segment_value):
    if segment_value == 0:
        return np.full_like(channel_values, np.nan, dtype=float)
    return np.asarray(channel_values, dtype=float) / segment_value


def segment_features(segment):
    mean_o_segment, std_o_segment, skew_o_segment, kurt_o_segment = compute_stats(segment)
    normalized_segment = normalize_per_channel(segment)
    mean_n_segment, std_n_segment, skew_n_segment, kurt_n_segment = compute_stats(normalized_segment)

    mean_o, std_o, skew_o, kurt_o = compute_stats(segment, axis=1)
    mean_n, std_n, skew_n, kurt_n = compute_stats(normalized_segment, axis=1)

    return pd.DataFrame({
        'mean_o': mean_o, 'std_o': std_o, 'skew_o': skew_o, 'kurt_o': kurt_o,
        'mean_n': mean_n, 'std_n': std_n, 'skew_n': skew_n, 'kurt_n': kurt_n,
        'mean_o_ratio': compute_ratio(mean_o, mean_o_segment),
        'std_o_ratio': compute_ratio(std_o, std_o_segment),
        'skew_o_ratio': compute_ratio(skew_o, skew_o_segment),
        'kurt_o_ratio': compute_ratio(kurt_o, kurt_o_segment),
        'mean_n_ratio': compute_ratio(mean_n, mean_n_segment),
        'std_n_ratio': compute_ratio(std_n, std_n_segment),
        'skew_n_ratio': compute_ratio(skew_n, skew_n_segment),
        'kurt_n_ratio': compute_ratio(kurt_n, kurt_n_segment)
    })


def contiguous_ranges(mask):
    changes = np.diff(np.concatenate([[False], mask, [False]]).astype(np.int8))
    return zip(np.flatnonzero(changes == 1), np.flatnonzero(changes == -1))


In [ ]:
filterbank_file = your.Your(str(filterbank_file_path))
n_time_samples = 256
segment = filterbank_file.get_data(
    nstart=inspection_segment_index * n_time_samples,
    nsamp=n_time_samples
).T
if segment.shape != (filterbank_file.your_header.nchans, n_time_samples):
    raise ValueError('Inspection segment has an unexpected shape')

inspection_model = fitted_top3_models[inspection_model_name]
inspection_features = segment_features(segment)
inspection_scores = probability_scores(
    inspection_model['pipeline'],
    inspection_features.loc[:, inspection_model['feature_columns']]
)
inspection_mask = inspection_scores >= inspection_model['threshold']
masked_segment = segment.astype(float).copy()
masked_segment[inspection_mask] = np.nan

masked_cmap = plt.get_cmap('gray').copy()
masked_cmap.set_bad(color='white')
fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True)
axes[0].imshow(segment, aspect='auto', origin='lower', cmap='gray')
axes[0].set_title('Original spectrogram')
axes[0].set_xlabel('Time sample')
axes[0].set_ylabel('Frequency channel')
axes[1].imshow(masked_segment, aspect='auto', origin='lower', cmap=masked_cmap)
for start, stop in contiguous_ranges(inspection_mask):
    axes[1].axhspan(start - 0.5, stop - 0.5, color='red', alpha=0.35)
axes[1].set_title(f'Predicted NBRFI channels: {inspection_mask.sum()}')
axes[1].set_xlabel('Time sample')
fig.tight_layout()
plt.show()


## Select model


In [ ]:
selected_model_name = None

if selected_model_name not in fitted_top3_models:
    raise ValueError('Set selected_model_name to one inspected model')


## Save selected model


In [ ]:
if bundle_path.exists():
    raise FileExistsError(f'Selected model already exists: {bundle_path}')

selected_model = fitted_top3_models[selected_model_name]
bundle = {
    'model_name': selected_model_name,
    'feature_columns': selected_model['feature_columns'],
    'threshold': selected_model['threshold'],
    'pipeline': selected_model['pipeline'],
    'validation_metrics': selected_model['validation_metrics'],
    'test_metrics': selected_model['test_metrics']
}
joblib.dump(bundle, bundle_path)
print('Selected model:', selected_model_name)
print('Model file:', bundle_path)
